<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Trace Pi Agent with Langfuse" sidebarTitle: "Pi Agent" logo: "/images/integrations/pi-agent_icon.svg" description: "Add LLM observability to the Pi coding agent with Langfuse by installing the community-maintained pi-langfuse extension to trace every session, model call, and tool call." category: "Integrations" -->

# Trace Pi Agent with Langfuse

This guide shows how to integrate **Langfuse** with **Pi**, a minimal AI coding agent, using the community-maintained `pi-langfuse` extension. Once configured, every Pi session is sent to Langfuse as a single trace — including the prompt, agent workflow, model generations, tool calls, the final response, token usage, cost, and trace-level scores.

<!-- CALLOUT_START type: "info" emoji: "ℹ️" -->
**Community-maintained integration.** The [`pi-langfuse`](https://www.npmjs.com/package/pi-langfuse) extension is built and maintained by the community, not by the Langfuse team. If you run into issues, please report them on the [extension's repository](https://github.com/gooyoung/pi-langfuse).
<!-- CALLOUT_END -->

> **What is Pi?** [Pi](https://pi.dev) is a minimal, extensible AI coding agent. It supports 15+ providers and hundreds of models, and adapts to your workflows through extensions, skills, prompt templates, and themes.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open-source LLM engineering platform that helps teams trace, debug, and evaluate their LLM applications.

<!-- STEPS_START -->

## Step 1: Install Pi

Install the Pi coding agent (requires Node.js >= 22). Pick whichever method fits your setup:

```bash
# npm (or pnpm / bun)
npm install -g --ignore-scripts @earendil-works/pi-coding-agent

# or via the install script (macOS / Linux)
curl -fsSL https://pi.dev/install.sh | sh
```

Verify the install and log in to a model provider (e.g. OpenAI, Anthropic, or Google) so Pi can run sessions:

```bash
pi --version
pi   # then run /login inside the interactive session
```

## Step 2: Install the pi-langfuse extension

Use Pi's package manager to install the extension. This downloads the package and registers it in your Pi settings automatically:

```bash
pi install npm:pi-langfuse
```

Confirm it is registered:

```bash
pi list
```

The extension hooks into Pi's lifecycle: it opens one trace per user prompt (grouped by session) with a root `agent` observation, a `generation` per model request, and a `tool` observation per tool call — plus the final assistant output, token usage, cost, and trace-level scores such as tool-call count and success rate.

## Step 3: Configure your Langfuse credentials

Get your Langfuse API keys from your project settings ([Langfuse Cloud](https://langfuse.com/cloud) → Settings → API Keys, or your [self-hosted](https://langfuse.com/self-hosting) instance). You can provide them in any of three ways.

**Option A — Interactive setup (recommended).** Run Pi once with the extension loaded; on first run without credentials it prompts for your public key, secret key, and host, then saves them to `~/.pi/agent/pi-langfuse/config.json`. To re-run setup or check status later:

```text
/langfuse-setup    # (re)enter credentials
/langfuse-status   # show host, masked key, capture policy
/langfuse-test     # verify host + keys
```

**Option B — Environment variables.** Used when the config file is missing or incomplete:

```bash
export LANGFUSE_PUBLIC_KEY="pk-lf-..."
export LANGFUSE_SECRET_KEY="sk-lf-..."
export LANGFUSE_BASE_URL="https://cloud.langfuse.com"  # 🇪🇺 EU (or https://us.cloud.langfuse.com for 🇺🇸 US, or your self-hosted host)
```

**Option C — Persistent `config.json`** at `~/.pi/agent/pi-langfuse/config.json`:

```json
{
  "publicKey": "pk-lf-...",
  "secretKey": "sk-lf-...",
  "host": "https://cloud.langfuse.com",
  "privacyPreset": "full-debug"
}
```

> **Privacy:** `privacyPreset` controls what payloads are uploaded — `metadata-only`, `prompts-only`, `conversations` (inputs + outputs, no tool I/O), or `full-debug` (everything; the default). Fine-grained flags (`LANGFUSE_CAPTURE_INPUTS`, `LANGFUSE_CAPTURE_OUTPUTS`, `LANGFUSE_CAPTURE_TOOL_IO`, `LANGFUSE_CAPTURE_SYSTEM_PROMPT`, `LANGFUSE_CAPTURE_CWD`) override the preset. All payloads are redacted for common secrets and local paths before upload. Keep `config.json` private and never commit it.

## Step 4: Run a Pi session

Start Pi as you normally would. Traces are scoped to each user prompt — a single message may span multiple model turns and tool calls, all grouped under one trace.

```bash
pi -p "Read README.md and summarize what this repository does"
```

When tracing is active, Pi prints `📊 Langfuse: Tracing enabled` at startup.

## Step 5: View traces in Langfuse

Open [Langfuse](https://langfuse.com/cloud) to see a trace for each Pi session. Each trace includes the root agent step, model generations (with token usage and cost), tool calls with their inputs and outputs, the final response, and trace-level scores.

![Example Pi Agent trace in Langfuse](https://langfuse.com/images/cookbook/integration-pi-agent/pi-agent-example-trace.png)

<!-- TODO: upload a screenshot of your trace to public/images/cookbook/integration-pi-agent/pi-agent-example-trace.png and replace the example link below with a publicly shared trace URL -->
[Example trace in Langfuse](https://cloud.langfuse.com/project/cmpy5u3fb06o5ad0j04xohhmz/traces/8aff6b31932ec387a4bcff7a73b79888)

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more.mdx" -->